# Example 03 — Switch models

**Teaches:** Change architecture with one key: `modeling.model_selection`.

Thin Colab wrapper around the canonical CLI example:

```bash
python -m run_pipeline --config examples/03_switch_models/config.yaml --force-retrain
```

Config + toy data live in [`examples/03_switch_models/`](./). Full guide: [`docs/COLAB.md`](../../docs/COLAB.md).

[Open in Colab](https://colab.research.google.com/github/Oikiy/ML-MEM-Review/blob/main/examples/03_switch_models/Colab.ipynb)



## 1. Bootstrap (clone + install on Colab)


In [ ]:
# === COLAB BOOTSTRAP (safe locally) =========================================
import os
import sys
import subprocess
from pathlib import Path

def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def _run(cmd, **kwargs):
    print("+", " ".join(cmd) if isinstance(cmd, (list, tuple)) else cmd)
    subprocess.check_call(cmd, **kwargs)

def _find_repo_root(start: Path):
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "config.yaml").is_file() and (candidate / "src").is_dir():
            return candidate
    return None

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

IN_COLAB = _in_colab()
REPO_URL = os.environ.get("ML_MEM_REPO_URL", "https://github.com/Oikiy/ML-MEM-Review.git")
REPO_NAME = os.environ.get("ML_MEM_REPO_NAME", "ML-MEM-Review")
COLAB_ROOT = Path(os.environ.get("ML_MEM_COLAB_ROOT", f"/content/{REPO_NAME}"))

root = _find_repo_root(Path.cwd())
if root is None and IN_COLAB:
    root = _find_repo_root(Path("/content"))

if IN_COLAB and root is None:
    if not COLAB_ROOT.exists():
        try:
            _run(["git", "clone", "--depth", "1", REPO_URL, str(COLAB_ROOT)])
        except subprocess.CalledProcessError as exc:
            raise RuntimeError(
                "git clone failed (exit 128 usually means the GitHub repo is "
                "private or the URL is wrong). Make the repo public, set "
                "ML_MEM_REPO_URL to a fork you can access, or upload a zip to "
                f"{COLAB_ROOT}. Original error: {exc}"
            ) from exc
    else:
        print(f"Using existing clone at {COLAB_ROOT}")
    os.chdir(COLAB_ROOT)
    root = _find_repo_root(Path.cwd())

if root is None:
    raise RuntimeError(
        "Could not locate ML-MEM-Review (need config.yaml + src/). "
        "On Colab: re-run this cell after a successful clone. "
        "Locally: open from inside a full clone of the repository."
    )

os.chdir(root)
print(f"Repo root: {root}")

src = root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

if IN_COLAB:
    marker = root / ".colab_install_ok"
    force = os.environ.get("ML_MEM_FORCE_INSTALL", "").strip().lower() in {"1", "true", "yes"}
    if force or not marker.exists():
        _run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)])
        try:
            marker.write_text("ok\n", encoding="utf-8")
        except OSError:
            pass
    else:
        print("Skipping pip install (found .colab_install_ok). "
              "Set ML_MEM_FORCE_INSTALL=1 to reinstall.")
    print("Colab bootstrap done.")
else:
    print("Local session: repo root + src/ on path (no auto pip install).")

REPO_ROOT = root
# === end bootstrap ===========================================================



## What this config emphasizes

- model_selection: randomforest (try extratrees, ridge, …)
- Same toy regression data as example 01



## 2. Inspect `config.yaml`


In [ ]:
# Show the example config (edit the file under examples/ to experiment)
from pathlib import Path

CONFIG_REL = "examples/03_switch_models/config.yaml"
CONFIG_PATH = REPO_ROOT / CONFIG_REL
assert CONFIG_PATH.is_file(), f"Missing {CONFIG_PATH} — bootstrap/clone incomplete?"

print(CONFIG_PATH.read_text(encoding="utf-8"))



## 3. Run `run_pipeline`


In [ ]:
# Run the pipeline (artifacts written under examples/03_switch_models/)
import os
import subprocess
import sys
from pathlib import Path

cmd = [
    sys.executable, "-m", "run_pipeline",
    "--config", str(CONFIG_PATH),
    "--force-retrain",
    "--stage", "eda,model",
]
env = os.environ.copy()
src = str(Path(REPO_ROOT) / "src")
env["PYTHONPATH"] = src + os.pathsep + env.get("PYTHONPATH", "")
print("+", " ".join(cmd))
completed = subprocess.run(cmd, cwd=str(REPO_ROOT), env=env)
if completed.returncode != 0:
    raise RuntimeError(
        f"run_pipeline exited {completed.returncode}. "
        "Scroll up for the pipeline traceback (not this cell)."
    )
print("Done.")



## 4. Show output figures


In [ ]:
# Display figures produced under this example
from pathlib import Path
from IPython.display import display, Markdown, Image

fig_dir = REPO_ROOT / "examples" / "03_switch_models" / "outputs" / "figures"
if not fig_dir.is_dir():
    print(f"No figures yet at {fig_dir}")
else:
    pngs = sorted(fig_dir.glob("*.png"))
    print(f"Found {len(pngs)} figure(s) in {fig_dir}")
    for png in pngs:
        display(Markdown(f"### `{png.name}`"))
        display(Image(filename=str(png)))



## Try next

- Edit `examples/03_switch_models/config.yaml` in the Colab file browser, re-run the pipeline cell
- Sibling examples under [`examples/`](../)
- Full membrane tutorial: [`notebooks/Complete_notebook/Ml-MEM-Tut.ipynb`](../../notebooks/Complete_notebook/Ml-MEM-Tut.ipynb)

